# 03 · Optimización con algoritmo genético (Fase III, Persona B)

**Objetivo:** buscar hiperparámetros de Random Forest con un algoritmo genético (GA) y compararlo contra
(a) la configuración inicial de `02_modelos` y (b) una búsqueda aleatoria con el **mismo presupuesto**.

Reglas de honestidad: todo se evalúa con validación cruzada solo sobre la submuestra de entrenamiento;
el conjunto de prueba **no se toca** aquí (se usa una sola vez en la Fase IV).
Como la configuración ganadora se elige con esos mismos pliegues, su puntaje de CV es algo **optimista**.

In [ ]:
import sys, time, platform, os
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
import preprocesamiento as pp, modelos as mo, ga

MODO_RAPIDO = True   # True: prueba en 1-2 min. Cambia a False para la corrida final del informe.
os.makedirs("../reports", exist_ok=True)
print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| CPUs:", os.cpu_count(), "|", platform.platform())

In [ ]:
df = pd.read_csv("../data/processed/train_submuestra.csv")
if MODO_RAPIDO:
    df = df.sample(5000, random_state=pp.SEED)
X, y, _ = pp.separar_xy(df)
num_cols, cat_cols = pp.columnas_por_tipo(X, excluir=pp.COLUMNAS_REDUNDANTES)
log_cols = pp.detectar_columnas_log(X, num_cols)
pre = pp.construir_preprocesador(num_cols, cat_cols, log_cols)
print(X.shape, "| % ataque:", round(y.mean()*100, 2), "| columnas numéricas:", len(num_cols), "| log:", len(log_cols))
print("Combinaciones posibles en el espacio:", ga.tamano_espacio())

## 1. Configuración del GA

| Elemento | Valor |
|---|---|
| Individuo (cromosoma) | `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` |
| Aptitud | F1 de la clase ataque, media de 3 pliegues estratificados (dentro del Pipeline) |
| Selección | torneo de tamaño 3 |
| Cruce | uniforme (prob. 0,8) |
| Mutación | por gen (prob. 0,25), valor nuevo al azar del espacio |
| Elitismo | 1 individuo |
| Población inicial | aleatoria (no se incluye la configuración inicial, para no darle ventaja) |

In [ ]:
if MODO_RAPIDO:
    TAM_POB, GENERACIONES = 4, 2
else:
    TAM_POB, GENERACIONES = 8, 5

ev_ga = ga.EvaluadorRF(pre, X, y, k=3, semilla=pp.SEED)
res_ga = ga.algoritmo_genetico(ev_ga, tam_poblacion=TAM_POB, generaciones=GENERACIONES,
                               paciencia=None, semilla=pp.SEED)
print("\nMejor GA:", ga.describir(res_ga["mejor_config"]), "| F1 (CV 3 pliegues) =", round(res_ga["mejor_aptitud"], 4))

## 2. Búsqueda aleatoria con el mismo presupuesto

In [ ]:
ev_rs = ga.EvaluadorRF(pre, X, y, k=3, semilla=pp.SEED)
res_rs = ga.busqueda_aleatoria(ev_rs, n_evaluaciones=res_ga["evaluaciones"], semilla=pp.SEED)
print("Mejor aleatoria:", ga.describir(res_rs["mejor_config"]), "| F1 =", round(res_rs["mejor_aptitud"], 4))

## 3. Evolución del GA

In [ ]:
h = res_ga["historia"]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(h["generacion"], h["mejor"], "o-", label="Mejor de la generación")
ax.plot(h["generacion"], h["media"], "s--", label="Media de la población")
ax.plot(h["generacion"], h["mejor_global"], "-", alpha=.5, label="Mejor global")
ax.set_xlabel("Generación"); ax.set_ylabel("F1 (CV, 3 pliegues)")
ax.set_title("Evolución de la aptitud del algoritmo genético"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("../reports/fig_ga_evolucion.png", dpi=150, bbox_inches="tight"); plt.show()

tr, tg = ev_rs.tabla_registro(), ev_ga.tabla_registro()
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.step(tg["evaluacion"], tg["mejor_hasta_ahora"], where="post", label="Algoritmo genético")
ax.step(tr["evaluacion"], tr["mejor_hasta_ahora"], where="post", label="Búsqueda aleatoria")
ax.set_xlabel("Evaluaciones (entrenamientos con CV)"); ax.set_ylabel("Mejor F1 hasta el momento")
ax.set_title("GA vs búsqueda aleatoria (mismo presupuesto)"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("../reports/fig_ga_vs_aleatoria.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Comparación final con la misma validación que la tabla del compañero (5 pliegues)

Se reevalúan las tres configuraciones con `mo.evaluar_cv` (5 pliegues). Atención: las dos configuraciones
optimizadas se eligieron con estos mismos datos, así que su ventaja sobre la inicial es **optimista**.
El número definitivo sale de la prueba (Fase IV).

In [ ]:
configs = {"RF inicial (02_modelos)": ga.CONFIG_INICIAL,
           "RF búsqueda aleatoria": res_rs["mejor_config"],
           "RF algoritmo genético": res_ga["mejor_config"]}
filas = []
for nombre, cfg in configs.items():
    t0 = time.perf_counter()
    r = mo.evaluar_cv(ga.construir_pipeline_rf(pre, cfg), X, y, k=5, semilla=pp.SEED)
    fila = {"configuración": nombre, **{k: cfg[k] for k in ga.ESPACIO_RF}}
    fila.update({f"{m}_media": v for m, v in r["media"].items()})
    fila["f1_desv"] = r["desv"].get("f1", np.nan)
    fila["segundos"] = round(time.perf_counter() - t0, 1)
    filas.append(fila)
comparacion = pd.DataFrame(filas)
comparacion.round(4)

In [ ]:
sufijo = "_rapido" if MODO_RAPIDO else ""
comparacion.to_csv(f"../reports/tabla_ga_comparacion{sufijo}.csv", index=False)
res_ga["historia"].to_csv(f"../reports/ga_historia{sufijo}.csv", index=False)
pd.concat([tg.assign(metodo="GA"), tr.assign(metodo="Aleatoria")]).to_csv(f"../reports/ga_evaluaciones{sufijo}.csv", index=False)
print("Guardado en ../reports (sufijo:", repr(sufijo), ")")

## 5. Notas para el informe
- Reportar el entorno de la corrida (CPU, versiones) de la celda inicial; los tiempos no son comparables con la tabla de Colab si cambia la máquina.
- Si GA y búsqueda aleatoria quedan casi empatados, decirlo: con un espacio de 1 584 combinaciones y pocas evaluaciones, la diferencia puede ser ruido (una sola semilla).
- Limitación: una sola ejecución por método; repetir con más semillas daría un intervalo, pero no hay tiempo.